# Data loading

In [4]:
import pandas as pd
import pickle
from pathlib import Path

# Get the Code directory (project root)
current_dir = Path.cwd()  # from_scratch directory
code_dir = current_dir.parent.parent.parent.parent  # Go up to Code directory
print(f"Code directory: {code_dir}")

# Define all data paths directly
PATHS = {
    # Training features
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    
    # Training targets
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    
    # Validation and test features
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    
    # Validation and test targets
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

# Print all paths for verification
print("\nData paths:")
for key, path in PATHS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {key}: {path}")

# Load all data
def load_all_data():
    """Load all data files"""
    data = {}
    
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    # Load parquet files
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    # Load pickle files
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    return data

# Load the data
data = load_all_data()

if data:
    print(f"\n" + "="*50)
    print(f"Successfully loaded {len(data)} datasets")
    print("="*50)
    for key, value in data.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {len(value)} samples")
else:
    print("\nNo data was loaded")

ImportError: Unable to import required dependencies:
numpy: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

# Clarans predefined

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn_extra.cluster import CLARANS
from sklearn.decomposition import PCA

# Set plot style
sns.set_theme(style="whitegrid")

def run_predefined_clarans(data_dict, k_clusters=3):
    """
    Uses the optimized sklearn-extra implementation of CLARANS.
    """
    datasets_to_run = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek']
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    
    results = {}

    print("\n" + "="*60)
    print("STARTING PREDEFINED CLARANS PIPELINE (SKLEARN-EXTRA)")
    print("="*60)

    for idx, key in enumerate(datasets_to_run):
        if key not in data_dict:
            continue
            
        print(f"\n--- Processing: {key} ---")
        X = data_dict[key]
        
        # Ensure data is in a format sklearn likes (float64 numpy array)
        X_vals = X.values.astype(np.float64)

        # 1. Initialize Predefined CLARANS
        # number_of_local_minima: similar to numlocal
        # max_neighbors: similar to maxneighbor
        model = CLARANS(
            n_clusters=k_clusters, 
            number_of_local_minima=3, 
            max_neighbors=100, 
            random_state=42
        )
        
        start_time = time.time()
        # 2. Fit and Predict
        labels = model.fit_predict(X_vals)
        duration = time.time() - start_time
        
        print(f"Done. Duration: {duration:.2f} seconds.")
        
        # 3. PCA for Visualization
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_vals)
        
        # Get medoids: model.medoid_indices_ gives the indices of the medoids
        medoids_pca = X_pca[model.medoid_indices_]
        
        # 4. Plotting
        ax = axes[idx]
        ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='viridis', s=15, alpha=0.5)
        
        # Plot the Medoids
        ax.scatter(medoids_pca[:, 0], medoids_pca[:, 1], c='red', marker='X', 
                   s=250, edgecolor='black', label='Medoids')
        
        ax.set_title(f"{key}\nLibrary CLARANS (Time: {duration:.2f}s)")
        ax.set_xlabel("PCA 1")
        ax.set_ylabel("PCA 2")
        ax.legend()

    plt.tight_layout()
    plt.show()
    return results

# Execute
if 'data' in globals():
    clustering_results = run_predefined_clarans(data, k_clusters=3)

ImportError: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

# Bayesian Search

In [ ]:
import numpy as np
import pandas as pd
import time
from sklearn_extra.cluster import CLARANS
from sklearn.metrics import silhouette_score
from bayes_opt import BayesianOptimization

# --- 1. Define the Objective Function ---

def clarans_cv_score(n_local_minima, max_neighbors):
    """
    Objective function for Bayesian Optimization.
    We want to maximize the Silhouette Score.
    """
    # Bayesian Optimization provides floats; CLARANS requires integers
    n_local_minima = int(round(n_local_minima))
    max_neighbors = int(round(max_neighbors))
    
    # We fix n_clusters to 3 as previously used, 
    # but you could also add it to the search space.
    model = CLARANS(
        n_clusters=3,
        number_of_local_minima=n_local_minima,
        max_neighbors=max_neighbors,
        random_state=42
    )
    
    try:
        # Fit and predict labels
        labels = model.fit_predict(X_search)
        
        # If the model fails to find more than one cluster, silhouette_score will fail
        if len(np.unique(labels)) < 2:
            return -1
            
        # Silhouette Score ranges from -1 to 1 (higher is better)
        score = silhouette_score(X_search, labels)
        return score
    except Exception as e:
        return -1

# --- 2. Prepare Data for Search ---

# We'll use the SMOTE-Tomek dataset
# Optimization is slow, so we take a sample of 2000 points for the search
X_full = data['X_train_smote_tomek'].values.astype(np.float64)
if len(X_full) > 2000:
    indices = np.random.choice(len(X_full), 2000, replace=False)
    X_search = X_full[indices]
else:
    X_search = X_full

# --- 3. Initialize and Run Bayesian Optimization ---

# Define the range for hyperparameters
# max_neighbors is usually a function of k(n-k), so we set a broad range
pbounds = {
    'n_local_minima': (2, 10),
    'max_neighbors': (20, 200)
}

optimizer = BayesianOptimization(
    f=clarans_cv_score,
    pbounds=pbounds,
    random_state=42,
    verbose=2
)

print(f"Starting Bayesian Search for CLARANS hyperparameters...")
start_time = time.time()

optimizer.maximize(
    init_points=5,  # Random exploration
    n_iter=10       # Targeted exploitation
)

end_time = time.time()

# --- 4. Results and Final Model ---

best_params = optimizer.max['params']
print("\n" + "="*50)
print(f"OPTIMIZATION COMPLETE in {end_time - start_time:.2f}s")
print(f"Best Silhouette Score: {optimizer.max['target']:.4f}")
print(f"Best n_local_minima: {int(round(best_params['n_local_minima']))}")
print(f"Best max_neighbors: {int(round(best_params['max_neighbors']))}")
print("="*50)

# Train the final model on the FULL dataset using optimized parameters
final_model = CLARANS(
    n_clusters=3,
    number_of_local_minima=int(round(best_params['n_local_minima'])),
    max_neighbors=int(round(best_params['max_neighbors'])),
    random_state=42
)

print("\nFitting final model on full SMOTE-Tomek dataset...")
final_labels = final_model.fit_predict(X_full)